# Evaluation on Kaggle (phase 3)

Thin wrapper (CLAUDE.md §4): installs the `t2sql` package from GitHub, rebuilds the dataset splits from the templates, then runs `t2sql.eval.run` once per model. No logic lives here.

**Before running**
- Settings → Accelerator: **GPU T4**; Internet: **On** (GitHub and the Hugging Face models).
- Add Input → the private dataset holding `eco2mix.duckdb` (it is not in git).

**After running**: download `predictions.zip` from the Output tab and unzip it into `results/` of the repository.

In [ ]:
REPO = "https://github.com/zaninip/domain-text2sql"
COMMIT = "main"  # pin a commit SHA to reproduce a run exactly
MODELS = ["Qwen/Qwen3-1.7B", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
SPLIT = "val"  # val for the model pilot; test for the baseline
MODE = "zero_shot"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[model]"

In [ ]:
import glob
import shutil
from pathlib import Path

found = glob.glob("/kaggle/input/**/eco2mix.duckdb", recursive=True)
assert found, "attach the dataset holding eco2mix.duckdb (Add Input)"
Path("data").mkdir(exist_ok=True)
shutil.copy(found[0], "data/eco2mix.duckdb")

In [ ]:
# Same command as `make dataset`; expected: train 2126 / val 402 / test 490
!python -m t2sql.dataset.build

In [ ]:
import subprocess

for model in MODELS:  # check=True stops the notebook if a run fails
    command = ["python", "-m", "t2sql.eval.run", "--model", model]
    subprocess.run([*command, "--split", SPLIT, "--mode", MODE], check=True)

In [ ]:
import json

keys = ["execution_accuracy", "valid_sql_rate", "mean_output_tokens"]
keys += ["generation_minutes", "outcomes"]
for path in sorted(Path("results/predictions").glob("*.summary.json")):
    summary = json.loads(path.read_text())
    print(summary["model"], summary["split"], summary["mode"], summary.get("device"))
    for key in keys:
        print(f"  {key}: {summary.get(key)}")

In [ ]:
!zip -qr /kaggle/working/predictions.zip results/predictions
!ls -lh /kaggle/working/predictions.zip